# 02 – Modellierungsstichprobe

Aus dem ungefilterten Panel (Notebook 01) wird hier die Stichprobe, auf der später geschätzt wird. Die Filterregeln sind **exakt** aus dem archivierten Modellierungscode übernommen (`archive/src/16_baseline_logit.py`, Funktion `load_sample`; der Ordner `archive/` wurde am 02.11.2026 gelöscht und liegt nur noch in der Git-Historie), nicht neu erfunden.

**Die Regeln, je in einem Satz:**
1. **Beobachtetes Label** (`h3_observed`): Nur Firmenjahre, bei denen die vollen 3 Folgejahre in den Daten liegen (FY ≤ 2022), weil wir sonst nicht wissen, ob die Firma noch Ziel wird.
2. **Finanzsektor raus** (`fin_sic_60_64`): Banken, Kreditinstitute, Broker und Versicherungen (SIC 60–64) fallen weg, weil ihre Bilanzen (Einlagen als Schulden) mit Industriefirmen nicht vergleichbar sind.
3. **Unsicherer Deal-Abschluss raus** (`completion_status == "uncertain"`): Firmen, bei denen nicht geklärt werden konnte, ob ihr Deal abgeschlossen wurde, sind weder saubere Positive noch saubere Negative und werden ganz ausgeschlossen.
4. **Blank-Check-Firmen raus** (SIC 6770): Mantelgesellschaften/SPACs haben kein operatives Geschäft und sind deshalb keine sinnvolle Vergleichsfirma.
5. **Größenfilter $50M Float mit IPO-Ausnahme**: Ein Firmenjahr bleibt nur, wenn sein eigener Streubesitzwert (`entity_public_float`) mindestens 50 Mio. USD beträgt – außer im ersten Börsenjahr mit Float 0 (`first_listing_year_exempt`), weil die Firma am Messtag noch gar nicht börsennotiert war (Messfehler, keine echte Null).

Hinweis: Der allgemeine Listing-Filter der Pipeline (Float > 0) ist schon beim Bau des Panels angewendet worden – alle Panel-Zeilen haben `passes_listing_filter = True`.

In [1]:
import pandas as pd
from pathlib import Path

DATA = Path("..") / "data" / "processed"
Y = "y_pe_main_h3"

# Laden und verbinden wie in Notebook 01
panel = pd.read_csv(DATA / "firm_year_panel.csv", dtype={"cik": str})
ratios = pd.read_csv(DATA / "firm_year_ratios.csv", dtype={"cik": str})
features = ["roa", "sales_growth_1y", "cash_ratio", "leverage", "log_assets",
            "float_to_book", "float_to_earnings"]
df = panel.merge(ratios[["accession"] + features], on="accession", how="inner", validate="one_to_one")
print(f"Verbunden: {len(df):,} Zeilen")
print("passes_listing_filter überall True:", df["passes_listing_filter"].all())

Verbunden: 71,713 Zeilen
passes_listing_filter überall True: True


## Trichter: Zeilen und Positive nach jedem Filter

Die Filter werden nacheinander angewendet; jede Zeile der Tabelle zeigt, was nach dem jeweiligen Schritt übrig ist. Positive werden nur unter `h3_observed`-Zeilen gezählt (vorher ist das Label nicht definiert).

In [2]:
steps = []                       # hier sammeln wir eine Zeile pro Filterschritt

def record(name, data, count_pos=True):
    """Merkt sich Zeilen, Firmen und Positive nach einem Filterschritt."""
    steps.append({
        "Schritt": name,
        "Zeilen": len(data),
        "Firmen": data["cik"].nunique(),
        "Positive (main)": int(data[Y].sum()) if count_pos else None,
        "positive Firmen": data.loc[data[Y] == 1, "cik"].nunique() if count_pos else None,
    })

record("0. Panel (verbunden)", df, count_pos=False)

s = df[df["h3_observed"]]
record("1. h3_observed", s)

s = s[~s["fin_sic_60_64"].astype(bool)]
record("2. ohne Finanzsektor (SIC 60-64)", s)

s = s[s["completion_status"] != "uncertain"]
record("3. ohne 'uncertain' Deal-Status", s)

# SIC ist als Zahl gespeichert, aber sicherheitshalber umwandeln (wie im Originalcode)
s = s[~pd.to_numeric(s["sic"], errors="coerce").isin([6770])]
record("4. ohne Blank-Check (SIC 6770)", s)

big_enough = s["entity_public_float"] >= 50e6
ipo_exempt = s["first_listing_year_exempt"].astype(bool)
s = s[big_enough | ipo_exempt]
record("5. Float >= $50M (oder IPO-Ausnahme)", s)

sample = s.copy()
funnel = pd.DataFrame(steps).set_index("Schritt")
# ganze Zahlen anzeigen; leere Felder (Schritt 0) bleiben leer statt NaN
funnel[["Positive (main)", "positive Firmen"]] = funnel[["Positive (main)", "positive Firmen"]].astype("Int64")
funnel["Anteil Positive"] = (funnel["Positive (main)"] / funnel["Zeilen"]).map(
    lambda v: "" if pd.isna(v) else f"{float(v):.2%}")
funnel

,Zeilen,Firmen,Positive (main),positive Firmen,Anteil Positive
Schritt,,,,,
0. Panel (verbunden),71713,11101,<NA>,<NA>,
1. h3_observed,61240,10743,327,136,0.53%
2. ohne Finanzsektor (SIC 60-64),52737,9455,311,130,0.59%
3. ohne 'uncertain' Deal-Status,52548,9419,311,130,0.59%
4. ohne Blank-Check (SIC 6770),50796,8638,310,130,0.61%
5. Float >= $50M (oder IPO-Ausnahme),33428,6466,282,119,0.84%


In [3]:
# Wie viele Zeilen hält nur die IPO-Ausnahme in der Stichprobe?
print(f"Zeilen nur wegen IPO-Ausnahme behalten: {(ipo_exempt & ~big_enough).sum():,}")

Zeilen nur wegen IPO-Ausnahme behalten: 1,429


**Ergebnis:** Von 61.240 beobachteten Firmenjahren bleiben nach allen Filtern 33.428 (6.466 Firmen) mit 282 positiven Zeilen von 119 Firmen. Der Größenfilter ist der mit Abstand stärkste Schritt: Er entfernt rund ein Drittel der Zeilen, aber nur 28 Positive – Take-Private-Ziele sind also kaum Kleinstfirmen. Deshalb steigt der Anteil der Positiven von 0,53 % auf 0,84 %. Die IPO-Ausnahme hält 1.429 Zeilen in der Stichprobe, die sonst wegen Float 0 herausfallen würden.

## Abgleich mit den erwarteten Zahlen

Erwartet: ca. 27.654 Zeilen, 281 bzw. 282 Positive, davon 167 im Training (2012–17). Die Zahlen werden hier nur verglichen, **nicht** angepasst.

In [4]:
# Aufteilung nach Zeitabschnitt (wie im Design: Train / Embargo / Test)
def split_of(year):
    if 2012 <= year <= 2017:
        return "Train 2012-17"
    if 2018 <= year <= 2019:
        return "Embargo 2018-19"
    return "Test 2020-22"

sample["split"] = sample["fiscal_year"].map(split_of)
by_split = sample.groupby("split", sort=False).agg(
    Zeilen=("accession", "size"),
    Positive=(Y, "sum"),
    positive_Firmen=("cik", lambda c: c[sample.loc[c.index, Y] == 1].nunique()))
by_split = by_split.loc[["Train 2012-17", "Embargo 2018-19", "Test 2020-22"]]
display(by_split)

train_test = sample[sample["split"] != "Embargo 2018-19"]
checks = pd.DataFrame([
    {"Größe": "Zeilen gesamt (alle Filter)",        "erwartet": "-",       "gefunden": len(sample)},
    {"Größe": "Zeilen Train + Test (ohne Embargo)", "erwartet": "ca. 27,654", "gefunden": len(train_test)},
    {"Größe": "Positive gesamt",                    "erwartet": "281 / 282", "gefunden": int(sample[Y].sum())},
    {"Größe": "Positive Training 2012-17",          "erwartet": "167",     "gefunden": int(by_split.loc["Train 2012-17", "Positive"])},
]).set_index("Größe")
checks

,Zeilen,Positive,positive_Firmen
split,,,
Train 2012-17,18058,167,76
Embargo 2018-19,5774,45,32
Test 2020-22,9596,70,43


,erwartet,gefunden
Größe,,
Zeilen gesamt (alle Filter),-,33428
Zeilen Train + Test (ohne Embargo),"ca. 27,654",27654
Positive gesamt,281 / 282,282
Positive Training 2012-17,167,167


**Ergebnis:** Alles stimmt mit dem archivierten Code überein. Die erwarteten **27.654 Zeilen** sind Training plus Test *ohne* die Embargo-Jahre 2018–19; mit Embargo sind es 33.428 (genau die Zahl in den alten Logs). **282 Positive** ist der aktuelle Label-Stand; die **281** in den alten Logs für Logit/Elastic Net/RF stammen von vor der letzten manuellen Prüfung, bei der Paramount Global (FY2022) positiv wurde – das betrifft nur ein Testjahr. Im Training sind es wie erwartet **167 Positive von 76 Firmen**.

## Kennzahlen: ungefiltert vs. gefiltert (Median, Positive vs. Negative)

Links wie in Notebook 01 (alle `h3_observed`-Zeilen), rechts die gefilterte Stichprobe (alle Abschnitte inkl. Embargo, damit es dieselbe Zeitspanne ist).

In [5]:
unfiltered = df[df["h3_observed"]]

def median_table(data):
    """Median jeder Kennzahl, getrennt für spätere Ziele (1) und Nicht-Ziele (0)."""
    t = data.groupby(Y)[features].median().T
    return t.rename(columns={1: "Positive", 0: "Negative"})[["Positive", "Negative"]]

compare = pd.concat({"ungefiltert": median_table(unfiltered),
                     "gefiltert": median_table(sample)}, axis=1).round(3)

# Zur Einordnung von log_assets: Median-Bilanzsumme in Mio. USD (= e hoch log_assets)
import numpy as np
assets_musd = np.exp(compare.loc["log_assets"]) / 1e6
display(compare)
print("Median-Bilanzsumme in Mio. USD:")
print(assets_musd.round(0).to_string())

ungefiltert          gefiltert         
y_pe_main_h3         Positive Negative  Positive Negative
roa                     0.015    0.003     0.016    0.020
sales_growth_1y         0.070    0.061     0.080    0.069
cash_ratio              0.068    0.088     0.068    0.094
leverage                0.244    0.127     0.264    0.201
log_assets             20.583   19.717    20.800   20.762
float_to_book           1.486    1.224     1.694    1.983
float_to_earnings       5.788    1.569     8.901    9.123

Median-Bilanzsumme in Mio. USD:
             y_pe_main_h3
ungefiltert  Positive         869.0
             Negative         366.0
gefiltert    Positive        1080.0
             Negative        1039.0


In [6]:
# float_to_earnings wird bei Verlust negativ -> Verlustanteil zum Lesen des Medians
loss = pd.DataFrame({
    "ungefiltert": unfiltered.groupby(Y)["float_to_earnings"].apply(lambda v: (v < 0).mean()),
    "gefiltert": sample.groupby(Y)["float_to_earnings"].apply(lambda v: (v < 0).mean()),
}).rename(index={1: "Positive", 0: "Negative"}).loc[["Positive", "Negative"]]
print("Anteil Zeilen mit Verlust:")
loss.map("{:.1%}".format)

Anteil Zeilen mit Verlust:


,ungefiltert,gefiltert
y_pe_main_h3,,
Positive,36.4%,35.5%
Negative,46.0%,36.7%


**Ergebnis:** Nach dem Filtern verschwindet der Größenunterschied fast ganz: Ziele und Nicht-Ziele haben im Median rund 1,1 bzw. 1,0 Mrd. USD Bilanzsumme. Der Unterschied in Notebook 01 kam also daher, dass unter den Nicht-Zielen viele Kleinstfirmen waren – nicht daher, dass PE große Firmen bevorzugt. Die **Verschuldung bleibt höher** (26 % statt 20 % der Bilanzsumme), aber der Abstand ist kleiner geworden. Neu sichtbar: Ziele sind im Median etwas **weniger profitabel** (ROA 1,6 % gegen 2,0 %), halten weniger Cash und sind **niedriger bewertet** (Float/Buchwert 1,7 gegen 2,0). Bei `float_to_earnings` sind die Mediane jetzt fast gleich, weil auch die Verlustanteile fast gleich sind (36 % gegen 37 %).

## Kernergebnisse

1. Mit genau den Regeln aus dem archivierten Code entsteht eine Stichprobe von 33.428 Firmenjahren mit 282 Positiven (119 Firmen); ohne Embargo sind es die erwarteten 27.654 Zeilen, im Training 167 Positive.
2. Der $50M-Float-Filter ist der wichtigste Schritt: Er wirft ein Drittel der Zeilen raus, aber kaum Ziele, sodass der Anteil der Positiven auf 0,84 % steigt.
3. **Größer?** Nein – in der gefilterten Stichprobe sind Ziele im Median praktisch genauso groß wie andere Firmen; der Unterschied in Notebook 01 war ein Effekt der Kleinstfirmen.
4. **Höher verschuldet?** Ja – Ziele haben weiterhin mehr Schulden (26 % gegen 20 % der Bilanzsumme), wenn auch mit kleinerem Abstand als ungefiltert.
5. Außerdem sind Ziele etwas niedriger bewertet, etwas weniger profitabel und halten weniger Cash – das passt zu Palepus Ideen von Unterbewertung und schwächerem Management, muss aber erst im Modell (mit allen Variablen gleichzeitig) geprüft werden.